# Lab 2 — What Did the Campaign Deliver?
Name: Andrii Bohdan | Group: KM-43

## Setup

In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab1-tv-panel/versions/3")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
commercials = pd.read_parquet(f"{DATA}/commercials.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")

### Preparing the data
Same preparation as in Lab 1: one datetime type for timestamps and dates, a person table (age, sex), monthly weights and audiences.

In [ ]:
import numpy as np


def to_naive(s):
    s = pd.to_datetime(s)
    if getattr(s.dt, "tz", None) is not None:
        s = s.dt.tz_localize(None)
    return s


for col in ["start_ts", "end_ts", "broadcast_date"]:
    programs[col] = to_naive(programs[col])
programs["broadcast_date"] = programs["broadcast_date"].dt.normalize()
for col in ["start_ts", "end_ts"]:
    viewings[col] = to_naive(viewings[col])
daily_weights["broadcast_date"] = to_naive(daily_weights["broadcast_date"]).dt.normalize()
commercials["start_ts"] = to_naive(commercials["start_ts"])

# person table: persons are R (101) and L (102); age = code 300, sex = codes 201/202
va = viewer_attributes.merge(attributes, on="attr_id", how="left")
code_name = attributes.set_index("attr_id")["attr_name"]
person_ids = va.loc[va["attr_id"].isin([101, 102]), "respondent_id"].unique()
age = (va[va["attr_id"] == 300][["respondent_id", "value"]]
       .rename(columns={"value": "age"}).drop_duplicates("respondent_id"))
sex = va[va["attr_id"].isin([201, 202])][["respondent_id", "attr_id"]].drop_duplicates("respondent_id")
sex["sex"] = sex["attr_id"].map(code_name).astype(str).str.strip().str[0].str.upper().replace({"F": "W"})
persons = (pd.DataFrame({"respondent_id": person_ids})
           .merge(age, on="respondent_id", how="left")
           .merge(sex[["respondent_id", "sex"]], on="respondent_id", how="left"))

# daily weights of persons (R and L only) and monthly weight = average over in-tab days
pdw = daily_weights[daily_weights["resp_type"].isin(["R", "L"])]
monthly_w = pdw.groupby("respondent_id")["weight"].mean()

AUDIENCES = {"P2+": (2, 200, None), "A18-49": (18, 49, None), "W18-34": (18, 34, "W")}


def members(aud):
    lo, hi, sx = AUDIENCES[aud]
    p = persons[persons["age"].between(lo, hi)]
    if sx is not None:
        p = p[p["sex"] == sx]
    return set(p["respondent_id"])


AUD_IDS = {a: members(a) for a in AUDIENCES}
BIG = ["CBS", "NBC", "ABC", "FOX", "FNC"]
SMALL = ["FX", "BRAVO", "HGTV", "FOOD", "HIST"]
NETS = BIG + SMALL

## Task 1.1 — Size

In [ ]:
print("Spots:", len(commercials))
print("Pods:", commercials["pod_id"].nunique())
print("Brands:", commercials["brand"].nunique())
print("Categories:", commercials["category"].nunique())
camp = commercials[commercials["brand"] == "BRAND_A"].copy()
print("BRAND_A spots:", len(camp))
display(camp["network_code"].value_counts().reindex(NETS, fill_value=0)
        .rename("BRAND_A spots").to_frame())

## Task 1.2 — Ad load

In [ ]:
c = commercials.copy()
c["clock_hour"] = c["start_ts"].dt.floor("h")
per_hour = c.groupby(["network_code", "clock_hour"])["length_sec"].sum() / 60
load = (per_hour.groupby("network_code").agg(["mean", "size"])
        .rename(columns={"mean": "Ad minutes per hour", "size": "Hours with ads"})
        .reindex(NETS))
load["Ad minutes per hour"] = load["Ad minutes per hour"].round(1)
display(load.sort_values("Ad minutes per hour", ascending=False))
print("Heaviest ad load:", load["Ad minutes per hour"].idxmax())

**Answer:** The broadcast networks (CBS, NBC, ABC, FOX) do not program the whole day nationally. Part of the schedule (local news, syndicated shows, early morning and late night) is handed over to the affiliated local stations, and the ad breaks in those hours are sold by the stations to local advertisers. They are not national commercials, so they are not in this log. Cable networks feed one national schedule all day, so almost every hour carries national ads (except paid programming and the VP debate, which had no ads at all).

## Task 1.3 — Price everything

In [ ]:
def hhmm_to_min(s):
    """'HH:MM' on the broadcast clock -> minutes since 06:00 of the broadcast day."""
    h, m = s.str.split(":", expand=True).astype(int).T.values
    return h * 60 + m - 360


rc = rate_card.copy()
rc["program_name"] = rc["program_name"].fillna("").astype(str).str.strip()
is_prog = rc["program_name"] != ""
rc_prog = rc[is_prog][["network_code", "program_name", "daypart", "cost_30s_usd"]].drop_duplicates(
    ["network_code", "program_name"])
rc_time = rc[~is_prog].copy()
rc_time["start_min"] = hhmm_to_min(rc_time["start_time"])
rc_time["end_min"] = hhmm_to_min(rc_time["end_time"])
rc_time = rc_time.explode("weekdays")
rc_time["weekdays"] = rc_time["weekdays"].astype(int)

spots = commercials.merge(programs[["telecast_id", "program_name"]], on="telecast_id", how="left")
bts = spots["start_ts"] - pd.Timedelta("6h")
spots["bdate"] = bts.dt.normalize()
spots["weekday"] = bts.dt.dayofweek
spots["minute"] = (bts - bts.dt.normalize()).dt.total_seconds() / 60

# 1) program rows win
spots = spots.merge(rc_prog, on=["network_code", "program_name"], how="left")
spots["row_kind"] = np.where(spots["daypart"].notna(), "program", "time")

# 2) time rows for the rest
rest = spots[spots["daypart"].isna()][["ad_airing_id", "network_code", "weekday", "minute"]]
m = rest.merge(rc_time[["network_code", "weekdays", "start_min", "end_min", "daypart", "cost_30s_usd"]],
               left_on=["network_code", "weekday"], right_on=["network_code", "weekdays"])
m = m[(m["start_min"] <= m["minute"]) & (m["minute"] < m["end_min"])]
print("Spots matched by more than one time row:", int(m["ad_airing_id"].duplicated().sum()))
m = m.drop_duplicates("ad_airing_id").set_index("ad_airing_id")
idx = spots["daypart"].isna()
spots.loc[idx, "daypart"] = spots.loc[idx, "ad_airing_id"].map(m["daypart"])
spots.loc[idx, "cost_30s_usd"] = spots.loc[idx, "ad_airing_id"].map(m["cost_30s_usd"])

spots["cost"] = spots["cost_30s_usd"] * spots["length_sec"] / 30   # :15 = half, :60 = double
print("Spots without a price:", int(spots["cost"].isna().sum()))

value = spots.groupby("network_code").agg(value=("cost", "sum"))
value["Rate-card value (USD M)"] = (value["value"] / 1e6).round(1)
prog_value = spots[spots["row_kind"] == "program"].groupby("network_code")["cost"].sum()
value["Share from program rows"] = (prog_value.reindex(value.index).fillna(0) / value["value"]).round(3)
value = value.drop(columns="value").reindex(NETS)
display(value)
print("Total rate-card value (USD M):", round(value["Rate-card value (USD M)"].sum(), 1))

## Task 2.1 — Delivery

In [ ]:
# the campaign spots with their price and broadcast day
camp = spots[spots["brand"] == "BRAND_A"].reset_index(drop=True)
camp["group"] = np.where(camp["network_code"].isin(BIG), "big", "small")

# exposures: a person saw a spot if watching that network at the spot's start (start <= t < end)
pv = viewings[viewings["resp_type"].isin(["R", "L"])]
rows = []
for net, sp in camp.groupby("network_code"):
    v = pv[pv["network_code"] == net]
    vs, ve, vid = v["start_ts"].values, v["end_ts"].values, v["respondent_id"].values
    for aid, t in zip(sp["ad_airing_id"].values, sp["start_ts"].values):
        hit = (vs <= t) & (t < ve)
        rows.append(pd.DataFrame({"ad_airing_id": aid, "respondent_id": vid[hit]}))
exposures = pd.concat(rows, ignore_index=True).drop_duplicates()

# weight of each viewer on the spot's broadcast day (in-tab persons only)
exposures = exposures.merge(camp[["ad_airing_id", "network_code", "group", "bdate"]], on="ad_airing_id")
exposures = exposures.merge(pdw[["respondent_id", "broadcast_date", "weight"]],
                            left_on=["respondent_id", "bdate"], right_on=["respondent_id", "broadcast_date"],
                            how="inner").drop(columns="broadcast_date")
print("Exposure rows:", len(exposures))


def universe_by_day(aud):
    d = pdw[pdw["respondent_id"].isin(AUD_IDS[aud])]
    return d.groupby("broadcast_date")["weight"].sum()


def spot_table(aud):
    """Impressions and rating of every campaign spot for one audience."""
    e = exposures[exposures["respondent_id"].isin(AUD_IDS[aud])]
    imp = e.groupby("ad_airing_id")["weight"].sum().reindex(camp["ad_airing_id"]).fillna(0).values
    uni = universe_by_day(aud).reindex(camp["bdate"]).values
    return pd.DataFrame({"impressions": imp, "rating": imp / uni * 100}, index=camp["ad_airing_id"])


def reach(aud, spot_ids=None):
    ids = AUD_IDS[aud]
    w = monthly_w[monthly_w.index.isin(ids)]
    e = exposures[exposures["respondent_id"].isin(ids)]
    if spot_ids is not None:
        e = e[e["ad_airing_id"].isin(spot_ids)]
    return w.reindex(e["respondent_id"].unique()).sum() / w.sum() * 100


delivery = {}
for aud in ["P2+", "A18-49"]:
    st = spot_table(aud)
    camp[f"imp_{aud}"] = st["impressions"].values
    camp[f"rating_{aud}"] = st["rating"].values
    grps = st["rating"].sum()
    r = reach(aud)
    delivery[aud] = dict(imp=st["impressions"].sum(), grps=grps, reach=r, freq=grps / r)

print("P2+ impressions (M):", f"{delivery['P2+']['imp'] / 1e6:.1f}")
print("P2+ GRPs:", f"{delivery['P2+']['grps']:.1f}")
print("P2+ reach (%):", f"{delivery['P2+']['reach']:.1f}")
print("P2+ average frequency:", f"{delivery['P2+']['freq']:.1f}")
print("A18-49 impressions (M):", f"{delivery['A18-49']['imp'] / 1e6:.1f}")
print("A18-49 GRPs:", f"{delivery['A18-49']['grps']:.1f}")
print("A18-49 reach (%):", f"{delivery['A18-49']['reach']:.1f}")
print("A18-49 average frequency:", f"{delivery['A18-49']['freq']:.1f}")

## Task 2.2 — Frequency distribution

In [ ]:
e_p2 = exposures[exposures["respondent_id"].isin(AUD_IDS["P2+"])]
freq = e_p2.groupby("respondent_id")["ad_airing_id"].nunique()
w_f = monthly_w.reindex(freq.index).dropna()
freq = freq[w_f.index]
dist = w_f.groupby(freq).sum() / 1e6

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(dist.index, dist.values, color=["#e07b00" if f >= 5 else "#1f5aa6" for f in dist.index])
ax.set_xlabel("Number of BRAND_A spots seen")
ax.set_ylabel("Projected persons, M")
ax.set_title("BRAND_A campaign: P2+ frequency distribution (orange = 5+ exposures)")
ax.grid(alpha=0.3, axis="y")
plt.show()

heavy = freq[freq >= 5].index
share5 = e_p2.loc[e_p2["respondent_id"].isin(heavy), "weight"].sum() / e_p2["weight"].sum()
print("Share of persons reached who saw 5+ spots:", f"{w_f[freq >= 5].sum() / w_f.sum():.3f}")
print("Share of impressions at 5+ exposures:", f"{share5:.3f}")

**Answer:** No. The distribution is strongly skewed: most of the people reached saw the ad only once or twice, while a small group of heavy TV viewers saw it many times. That small 5+ group takes a share of all impressions far larger than its share of people (see the numbers above). The average frequency mixes these two groups and describes almost nobody: the typical person reached saw fewer spots than the average, and the heavy viewers saw many more. For planning, the distribution (or the effective reach at 3+, and the share of impressions wasted on 5+) tells much more than the single average.

## Task 2.3 — On target?

In [ ]:
target_share = delivery["A18-49"]["imp"] / delivery["P2+"]["imp"]
pop_share = (universe_by_day("A18-49") / universe_by_day("P2+")).mean()
print("Share of impressions on A18-49:", f"{target_share:.3f}")
print("A18-49 share of all persons:", f"{pop_share:.3f}")

by_net = camp.groupby("network_code")[["imp_P2+", "imp_A18-49"]].sum()
by_net["A18-49 share"] = by_net["imp_A18-49"] / by_net["imp_P2+"]
by_net["Index vs population"] = (by_net["A18-49 share"] / pop_share * 100).round(0)
by_net["A18-49 share"] = by_net["A18-49 share"].round(3)
by_net["P2+ impressions (M)"] = (by_net["imp_P2+"] / 1e6).round(1)
by_net = by_net[["P2+ impressions (M)", "A18-49 share", "Index vs population"]].reindex(NETS)
display(by_net.sort_values("Index vs population"))

**Answer:** The target is about 42% of all persons, so an on-target plan would put about 42% of its impressions on A18–49 (index 100). Compare the printed share with that: if it is below, the plan under-delivered against the people it was bought for and part of the money reached older viewers. The table above explains it network by network: networks with an index below 100 have an older audience (typically Fox News, HGTV, History, the broadcast networks' daytime and news) and pull the share down, especially if they carry many of the spots; younger networks such as FX and Bravo (index above 100) pull it up but carry only a small part of the impressions.

## Task 3.1 — Cost

In [ ]:
cost = camp["cost"].sum()
p2, a = delivery["P2+"], delivery["A18-49"]
print("Campaign cost (USD):", f"{cost:.0f}")
print("P2+ CPM (USD):", f"{cost / (p2['imp'] / 1000):.2f}")
print("P2+ CPP (USD):", f"{cost / p2['grps']:.0f}")
print("A18-49 CPM (USD):", f"{cost / (a['imp'] / 1000):.2f}")
print("A18-49 CPP (USD):", f"{cost / a['grps']:.0f}")

## Task 3.2 — By network

In [ ]:
net = camp.groupby("network_code").agg(spots=("ad_airing_id", "size"), cost=("cost", "sum"),
                                       imp_p2=("imp_P2+", "sum"), imp_a=("imp_A18-49", "sum"))
net_table = pd.DataFrame({
    "Spots": net["spots"],
    "Cost (USD)": net["cost"].round(0).astype(int),
    "Cost share": (net["cost"] / net["cost"].sum()).round(3),
    "P2+ impressions (M)": (net["imp_p2"] / 1e6).round(1),
    "A18-49 impressions (M)": (net["imp_a"] / 1e6).round(1),
    "P2+ CPM (USD)": (net["cost"] / (net["imp_p2"] / 1000)).round(2),
    "A18-49 CPM (USD)": (net["cost"] / (net["imp_a"] / 1000)).round(2),
}).reindex(NETS)
display(net_table.sort_values("A18-49 CPM (USD)"))
print("Most money spent on:", net_table["Cost (USD)"].idxmax())
print("Cheapest network for A18-49 (CPM):", net_table["A18-49 CPM (USD)"].idxmin())

**Answer:** Most of the budget went to the big broadcast networks (see the cost share column): their spots are the most expensive on the rate card, especially sports and prime time sold by program rows. They also deliver most of the impressions, but not the cheapest ones. The cheapest way to reach A18–49 is the networks at the top of the table, with the lowest A18–49 CPM: networks whose spots are cheap and whose audience is young enough. Networks with an old audience (index below 100 in Task 2.3) have a high A18–49 CPM even if their P2+ CPM looks fine, because most of what the client pays for there is not the target.

## Task 3.3 — Guarantee

In [ ]:
GUARANTEE = 150
p2_cpp = cost / p2["grps"]
owed = max(0.0, GUARANTEE - p2["grps"])
print("Guarantee met:", bool(p2["grps"] >= GUARANTEE))
print("Make-good GRPs owed:", f"{owed:.1f}")
print("Value of make-goods (USD):", f"{owed * p2_cpp:.0f}")

## Task 4 — Do networks add up?

In [ ]:
big_ids = camp.loc[camp["group"] == "big", "ad_airing_id"]
small_ids = camp.loc[camp["group"] == "small", "ad_airing_id"]
r_big = reach("P2+", big_ids)
r_small = reach("P2+", small_ids)
r_all = reach("P2+")
overlap = r_big + r_small - r_all
overlap_ind = r_big * r_small / 100
added = r_all - r_big
cost_big = camp.loc[camp["group"] == "big", "cost"].sum()
cost_small = camp.loc[camp["group"] == "small", "cost"].sum()

print("Reach, big networks (%):", f"{r_big:.1f}")
print("Reach, small networks (%):", f"{r_small:.1f}")
print("Reach, whole campaign (%):", f"{r_all:.1f}")
print("Overlap (%):", f"{overlap:.1f}")
print("Overlap if independent (%):", f"{overlap_ind:.1f}")
print("Reach added by small networks (points):", f"{added:.1f}")
print("Cost per added reach point, small (USD):", f"{cost_small / added:.0f}" if added > 0 else "n/a")
print("Cost per reach point, big (USD):", f"{cost_big / r_big:.0f}")
print("Share of small-network reach already reached by big:", f"{overlap / r_small:.3f}")

**Answer:** Reach counts people, not exposures: a person who saw the ad on CBS and again on HGTV is counted once in the whole campaign but once in each group, so the sum of the two reaches counts the overlap twice (whole = big + small − overlap). If the two audiences were independent, the overlap would be big × small / 100. The actual overlap is larger than that (compare the two printed lines), so the small networks' viewers were more likely than average to have been reached by the big networks already: people who watch a lot of cable also watch a lot of TV in general, including the broadcast networks, so heavy viewers are reached by both groups while light viewers are missed by both. As a result, the small networks added only a few points of new reach, and each added point cost much more than a reach point on the big networks (see the two cost lines). Most of the small-network money bought extra frequency for people already reached, not new people; it was well spent only if the goal was frequency or a cheaper CPM, not reach.

## Task 5 — How sure are we?

In [ ]:
# unweighted panelists (R and L persons in-tab that day) who saw each spot
panelists = exposures.groupby("ad_airing_id")["respondent_id"].nunique().reindex(camp["ad_airing_id"]).fillna(0)
camp["panelists"] = panelists.values

# standard error of the P2+ rating: n = persons in-tab that broadcast day
p2_dw = pdw[pdw["respondent_id"].isin(AUD_IDS["P2+"])]
n_intab = p2_dw.groupby("broadcast_date")["respondent_id"].nunique()
camp["n"] = n_intab.reindex(camp["bdate"]).values
p = camp["rating_P2+"] / 100
camp["SE"] = np.sqrt(p * (1 - p) / camp["n"]) * 100
camp["rel_SE"] = np.where(camp["rating_P2+"] > 0, camp["SE"] / camp["rating_P2+"], np.nan)

t5 = camp.groupby("network_code").agg(
    spots=("ad_airing_id", "size"),
    median_panelists=("panelists", "median"),
    median_P2_rating=("rating_P2+", "median"),
    median_rel_SE=("rel_SE", "median"),
    zero_panelist_spots=("panelists", lambda s: int((s == 0).sum())),
).reindex(NETS).round(3)
display(t5)

# W18-34 zero-cells: spots no woman 18-34 in the panel saw
w1834 = exposures[exposures["respondent_id"].isin(AUD_IDS["W18-34"])]
zero_w = (~camp["ad_airing_id"].isin(w1834["ad_airing_id"])).sum()

print("Spots seen by no panelist:", int((camp["panelists"] == 0).sum()))
print("Median relative SE, big networks:", f"{camp.loc[camp['group'] == 'big', 'rel_SE'].median():.3f}")
print("Median relative SE, small networks:", f"{camp.loc[camp['group'] == 'small', 'rel_SE'].median():.3f}")
print("W18-34 zero-cell spots:", int(zero_w))
print("Persons in-tab, average day:", round(n_intab.mean(), 1))
print("Women 18-34 in-tab, average day:",
      round(pdw[pdw["respondent_id"].isin(AUD_IDS["W18-34"])].groupby("broadcast_date").size().mean(), 1))

**Answer:** A spot seen by no panelist gets a rating of exactly 0, but that does not mean nobody in the US saw it: it means none of the few thousand panelists happened to be watching. With each person standing for tens of thousands of Americans, one panelist more or less moves the rating a lot, so small-network spots, seen by only a handful of panelists, have zero ratings and large relative errors (often comparable to the rating itself), while big-network spots rest on many more panelists and are much more stable. The SE here is only a lower bound, since a weighted panel behaves like a smaller sample. For the W18–34 rating by spot I would tell the client that the panel cannot support it: many spots are zero-cells, and the non-zero ones rest on one or two women, so per-spot numbers are noise. I would report W18–34 only at the campaign level (total GRPs, reach) or by network over the whole month, where enough panelists add up, and treat it as an estimate with a wide error.

## Bonus

In [ ]:
# A18-49 CPM of the campaign's own spots, by network and selling daypart
bon = camp.groupby(["network_code", "daypart"]).agg(spots=("ad_airing_id", "size"), cost=("cost", "sum"),
                                                    imp_a=("imp_A18-49", "sum"))
bon["cost_per_spot"] = bon["cost"] / bon["spots"]
bon["A18-49 CPM"] = bon["cost"] / (bon["imp_a"] / 1000)
bon = bon[(bon["imp_a"] > 0) & (bon["spots"] >= 5)].sort_values("A18-49 CPM")   # 5+ spots: less panel noise
display(bon.round(2))

# move the budget of 20 spots from the most expensive inventory (by A18-49 CPM) to the cheapest
worst = camp.merge(bon[["A18-49 CPM"]], left_on=["network_code", "daypart"], right_index=True)
worst = worst.sort_values("A18-49 CPM", ascending=False).head(20)
budget = worst["cost"].sum()
best = bon.iloc[0]
lost_imp = worst["imp_A18-49"].sum()
gained_imp = budget / best["cost"] * best["imp_a"]
new_imp = a["imp"] - lost_imp + gained_imp
print("Budget moved (USD):", f"{budget:.0f}")
print("Moved from:", sorted(set(zip(worst["network_code"], worst["daypart"]))))
print("Moved to:", bon.index[0], "| spots bought:", round(budget / best["cost_per_spot"], 1))
print("A18-49 impressions (M), before -> after:", f"{a['imp'] / 1e6:.1f} -> {new_imp / 1e6:.1f}")
print("A18-49 CPM (USD), before -> after:", f"{cost / (a['imp'] / 1000):.2f} -> {cost / (new_imp / 1000):.2f}")

**Answer:** I would take the 20 spots with the highest A18–49 CPM (the network/daypart pairs printed under "Moved from": expensive inventory whose audience is mostly older than the target) and spend the same budget in the inventory with the lowest A18–49 CPM ("Moved to"). At the A18–49 cost per thousand this campaign actually achieved there, the same money buys more target impressions (the estimate uses only network/daypart cells with at least 5 campaign spots, because cells with 1–2 spots are pure panel noise), so the campaign's A18–49 CPM falls from the "before" to the "after" value printed above at the same total cost. Since the cheaper inventory is on a different network and daypart, it also reaches some young adults the original plan missed, which should raise A18–49 reach as well; the estimate assumes the new spots perform like the client's existing spots there.